# Violin plots of individual CLIP scores for quilt1m vs quilt1m_curated

Compares distributions of per-sample CLIP scores saved by retrieval evaluation.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Inputs from Snakemake
inputs = list(snakemake.input.individual_scores)
out_path = Path(snakemake.output.plot)
out_path.parent.mkdir(parents=True, exist_ok=True)
labels = snakemake.params.test_datasets

# Load and annotate
dfs = []
for csv, label in zip(inputs, labels):
    df = pd.read_csv(csv)
    df['dataset'] = label
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)

# Per-sample mean of both directions
df['clip_score_mean'] = (df['clip_score_left_right'] + df['clip_score_right_left']) / 2.0

# Style and plot
plt.style.use(str(snakemake.input.mpl_style))
fig, ax = plt.subplots(figsize=(6, 6))
sns.violinplot(data=df, x='dataset', y='clip_score_mean', inner='quartile', cut=0, ax=ax)
ax.set_title('Per-sample mean CLIP score (quilt vs curated)')
ax.set_xlabel('Test dataset')
ax.set_ylabel('Mean CLIP score (logits)')
fig.tight_layout()
fig.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
